In [1]:
import os
import re
import json
import pickle
import tqdm

from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

True

In [ ]:
SEED = 23
TEMPERATURE = 0.1

In [3]:
MODEL = "google/gemini-2.5-flash"

In [4]:
output_dir_name = MODEL.split("/")[1]

In [5]:
with open(f"{output_dir_name}/dbscan_clusters.pkl", "rb") as f:
    dbscan_clusters = pickle.load(f)

In [6]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can generalize a set of tags to one tag.
For example, for a givent list of tags return one general tag.
Input: ["economics", "macroeconomics", "monetary_policy", "forecasting", "time_series"]
Ouput: {"tag_name": "economics"}
Return strictly a JSON in a snake_case in English as an answer.
If you cannot generalize, return {"tag_name": "uncategorized"} 
"""

In [7]:
messages = []

In [8]:
for cluster_id in sorted(dbscan_clusters.keys())[1:]:
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(dbscan_clusters[cluster_id], ensure_ascii=False),
            },
        )
    )

In [9]:
tags_map = dict()

In [10]:
client = OpenAI(
    api_key=os.environ.get("KODIK_API_KEY"),
    base_url="https://api.kodikrouter.ru/v1",
)

In [ ]:
for message in tqdm.tqdm(messages[2:]):

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=message,
            seed=SEED,
            temperature=TEMPERATURE,
        )

        content = response.choices[0].message.content

        json_match = re.search(r"```json\s*(.*?)\s*```", content, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_str = content

        answer = json.loads(json_str)
        tag_name = answer["tag_name"]

        if tag_name != "uncategorized":
            for tag in json.loads(message[1]["content"]):
                tags_map[tag] = tag_name
        else:
            for tag in json.loads(message[1]["content"]):
                tags_map[tag] = tag

    except Exception as e:

        print(f'{message[1]["content"]} was not processed due to {e}')

 70%|██████▉   | 115/165 [01:59<00:44,  1.13it/s]

["machine_learning", "machine_learning_operations"] was not processed due to Error code: 429 - {'error': {'message': 'Превышен лимит запросов', 'type': 'rate_limit_error', 'param': None, 'code': 'rate_limited'}}


100%|██████████| 165/165 [03:12<00:00,  1.17s/it]


In [12]:
len(tags_map)

761

In [13]:
len(set(tags_map.values()))

160

In [14]:
with open(f"{output_dir_name}/titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags_dict = pickle.load(f)

In [15]:
with open(f"{output_dir_name}/tags_set.pkl", "rb") as f:
    tags_set = pickle.load(f)

In [16]:
titles_with_tags_dict_processed = {}
tags_set_processed = set()

for title in titles_with_tags_dict:
    data = set()
    for tag in titles_with_tags_dict[title]:
        tag = tags_map.get(tag, tag)
        data.add(tag)
        tags_set_processed.add(tag)
    titles_with_tags_dict_processed[title] = sorted(data)

In [17]:
if not os.path.exists(output_dir_name):
    os.mkdir(output_dir_name)

In [18]:
with open(f"{output_dir_name}/titles_with_tags_dict_processed.pkl", "wb") as f:
    pickle.dump(titles_with_tags_dict_processed, f)

In [19]:
with open(f"{output_dir_name}/tags_set_processed.pkl", "wb") as f:
    pickle.dump(tags_set_processed, f)

In [20]:
len(tags_set)

1524

In [21]:
len(tags_set_processed)

918

In [22]:
list(tags_set_processed)

['north_caucasus',
 'meiji_restoration',
 'promotion_campaigns',
 'online_banking',
 'user_interface_design',
 'beneficial_ownership',
 'graph_theory',
 'belarusian_foreign_policy',
 'job_creation',
 'compositional_analysis',
 'electromagnetism',
 'moral_psychology',
 'colonialism',
 'diagnosis',
 'linux_kernel',
 'place_attachment',
 'transportation',
 'climate_change',
 'climate_science',
 ' foresight',
 'virginia_woolf',
 'adaptation',
 'data_anonymization',
 'hebrew',
 'field_work',
 'peptide_science',
 'competition',
 'self_efficacy',
 'integrated_marketing_communications',
 'peer_support',
 'ethnic_processes',
 'law',
 'econometrics',
 'civic_education',
 'sociology_of_work',
 'dentistry',
 'automation',
 'hsee',
 'intergenerational_conflict',
 'self-improvement',
 'llm',
 'differential_geometry',
 'philosophy_of_literature',
 'generation_alpha',
 'statistical_methods',
 'adaptive_testing',
 'customer_churn',
 'cross_media',
 'steampunk',
 'brics',
 'music_therapy',
 'static_anal